# Mutual Fund Performance Analytics

Comprehensive performance analysis of 40 Indian mutual fund schemes:
- Daily Returns, CAGR, Sharpe/Sortino Ratios
- Alpha & Beta (OLS regression vs NIFTY 100)
- Maximum Drawdown, Fund Scorecard
- Benchmark Comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

REPO = Path().resolve().parent
RAW = REPO / 'data' / 'raw'
RF_ANNUAL = 0.065
RF_DAILY = RF_ANNUAL / 252

print('Setup complete')

In [ ]:
# Load data
nav = pd.read_csv(RAW / '02_nav_history.csv')
nav['date'] = pd.to_datetime(nav['date'])
fm = pd.read_csv(RAW / '01_fund_master.csv')
bench = pd.read_csv(RAW / '10_benchmark_indices.csv')
bench['date'] = pd.to_datetime(bench['date'])

nav_pivot = nav.pivot_table(index='date', columns='amfi_code', values='nav').sort_index()
daily_returns = nav_pivot.pct_change().iloc[1:]

print(f'NAV: {nav_pivot.shape[0]} days x {nav_pivot.shape[1]} schemes')
print(f'Daily returns: {daily_returns.shape[0]} days x {daily_returns.shape[1]} schemes')
print(f'Date range: {nav_pivot.index[0].date()} to {nav_pivot.index[-1].date()}')

## 1. Daily Returns Distribution

In [ ]:
all_returns = daily_returns.stack()
print(all_returns.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
all_returns.hist(bins=80, ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_title('Daily Returns Distribution (All 40 Schemes)')
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')

stats.probplot(all_returns, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot')
plt.tight_layout()
plt.savefig(REPO / 'reports' / 'daily_returns_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Distribution looks reasonable: mean near 0, slight positive skew, fat tails.')

## 2. CAGR (1yr, 3yr, 5yr)

In [ ]:
def compute_cagr(series, period_days):
    if len(series) < period_days:
        return np.nan
    end = series.iloc[-1]
    start = series.iloc[-period_days - 1]
    if start <= 0 or end <= 0:
        return np.nan
    n_years = period_days / 252
    return (end / start) ** (1 / n_years) - 1

cagr_data = []
for code in nav_pivot.columns:
    cagr_data.append({
        'amfi_code': code,
        'CAGR_1yr': compute_cagr(nav_pivot[code], 252),
        'CAGR_3yr': compute_cagr(nav_pivot[code], 252*3),
        'CAGR_5yr': compute_cagr(nav_pivot[code], 252*5),
    })

cagr_df = pd.DataFrame(cagr_data)
cagr_df = cagr_df.merge(fm[['amfi_code', 'scheme_name', 'fund_house']], on='amfi_code')
print('5yr CAGR not available (data < 5 years).')
cagr_df[['scheme_name', 'CAGR_1yr', 'CAGR_3yr']].head(10)

## 3. Sharpe Ratio (Rf = 6.5%)

In [ ]:
def sharpe_ratio(returns_series):
    excess = returns_series - RF_DAILY
    if returns_series.std() == 0:
        return np.nan
    return (excess.mean() / returns_series.std()) * np.sqrt(252)

sharpe_data = []
for code in daily_returns.columns:
    sharpe_data.append({'amfi_code': code, 'Sharpe': sharpe_ratio(daily_returns[code])})

sharpe_df = pd.DataFrame(sharpe_data)
sharpe_df = sharpe_df.merge(fm[['amfi_code', 'scheme_name', 'fund_house']], on='amfi_code')
sharpe_df.sort_values('Sharpe', ascending=False)[['scheme_name', 'Sharpe']].head(10)

## 4. Sortino Ratio (Downside Deviation)

In [ ]:
def sortino_ratio(returns_series):
    excess = returns_series - RF_DAILY
    downside = returns_series[returns_series < 0]
    if len(downside) == 0 or downside.std() == 0:
        return np.nan
    return (excess.mean() / downside.std()) * np.sqrt(252)

sortino_data = []
for code in daily_returns.columns:
    sortino_data.append({'amfi_code': code, 'Sortino': sortino_ratio(daily_returns[code])})

sortino_df = pd.DataFrame(sortino_data)
sortino_df = sortino_df.merge(fm[['amfi_code', 'scheme_name', 'fund_house']], on='amfi_code')
sortino_df.sort_values('Sortino', ascending=False)[['scheme_name', 'Sortino']].head(10)

## 5. Alpha and Beta (OLS vs NIFTY 100)

In [ ]:
nifty100 = bench[bench['index_name'] == 'NIFTY100'][['date', 'close_value']].copy()
nifty100 = nifty100.sort_values('date')
nifty100['return'] = nifty100['close_value'].pct_change()
nifty100 = nifty100.dropna().set_index('date')

common_dates = daily_returns.index.intersection(nifty100.index)
dr = daily_returns.loc[common_dates]
br = nifty100.loc[common_dates, 'return']

alpha_beta_records = []
for code in dr.columns:
    fund_ret = dr[code].dropna()
    bench_ret = br.loc[fund_ret.index]
    if len(fund_ret) < 30:
        continue
    slope, intercept, r_val, p_val, std_err = stats.linregress(bench_ret, fund_ret)
    alpha_beta_records.append({
        'amfi_code': code,
        'Alpha': intercept * 252,
        'Beta': slope,
        'R_squared': r_val ** 2,
        'p_value': p_val
    })

alpha_beta_df = pd.DataFrame(alpha_beta_records)
alpha_beta_df = alpha_beta_df.merge(fm[['amfi_code', 'scheme_name', 'fund_house']], on='amfi_code')
alpha_beta_df.sort_values('Alpha', ascending=False)[['scheme_name', 'Alpha', 'Beta', 'R_squared']].head(10)

In [ ]:
# Scatter plot: Beta vs Alpha
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(alpha_beta_df['Beta'], alpha_beta_df['Alpha'], 
                     c=alpha_beta_df['R_squared'], cmap='viridis', s=60, alpha=0.8)
for _, row in alpha_beta_df.iterrows():
    ax.annotate(row['scheme_name'][:25], (row['Beta'], row['Alpha']), fontsize=7, alpha=0.7)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Beta')
ax.set_ylabel('Alpha (annualized)')
ax.set_title('Alpha vs Beta (vs NIFTY 100)')
plt.colorbar(scatter, label='R²')
plt.tight_layout()
plt.savefig(REPO / 'reports' / 'alpha_beta_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Maximum Drawdown

In [ ]:
def max_drawdown(series):
    running_max = series.cummax()
    dd = series / running_max - 1
    return dd.min()

def max_dd_details(series):
    running_max = series.cummax()
    dd = series / running_max - 1
    dd_min_idx = dd.idxmin()
    peak_before = series[:dd_min_idx].idxmax()
    return peak_before, dd_min_idx, dd.min()

mdd_data = []
for code in nav_pivot.columns:
    peak, trough, val = max_dd_details(nav_pivot[code])
    mdd_data.append({
        'amfi_code': code,
        'Max_DD': val,
        'Peak_Date': peak.date(),
        'Trough_Date': trough.date()
    })

mdd_df = pd.DataFrame(mdd_data)
mdd_df = mdd_df.merge(fm[['amfi_code', 'scheme_name', 'fund_house']], on='amfi_code')
mdd_df.sort_values('Max_DD')[['scheme_name', 'Max_DD', 'Peak_Date', 'Trough_Date']].head(10)

## 7. Fund Scorecard (0–100)

In [ ]:
scorecard = pd.DataFrame({'amfi_code': daily_returns.columns.tolist()})
scorecard = scorecard.merge(cagr_df[['amfi_code', 'CAGR_3yr']], on='amfi_code')
scorecard = scorecard.merge(sharpe_df[['amfi_code', 'Sharpe']], on='amfi_code')
scorecard = scorecard.merge(alpha_beta_df[['amfi_code', 'Alpha']], on='amfi_code')
scorecard = scorecard.merge(mdd_df[['amfi_code', 'Max_DD']], on='amfi_code')
scorecard['expense_ratio_pct'] = scorecard['amfi_code'].map(fm.set_index('amfi_code')['expense_ratio_pct'])

for col in ['CAGR_3yr', 'Sharpe', 'Alpha']:
    scorecard[f'{col}_rank'] = scorecard[col].rank(ascending=False, na_option='bottom')
scorecard['expense_rank'] = scorecard['expense_ratio_pct'].rank(ascending=True, na_option='bottom')
scorecard['Max_DD_rank'] = scorecard['Max_DD'].rank(ascending=True, na_option='bottom')

n = len(scorecard)
scorecard['Score'] = (
    0.30 * (1 - scorecard['CAGR_3yr_rank'] / n) +
    0.25 * (1 - scorecard['Sharpe_rank'] / n) +
    0.20 * (1 - scorecard['Alpha_rank'] / n) +
    0.15 * (1 - scorecard['expense_rank'] / n) +
    0.10 * (1 - scorecard['Max_DD_rank'] / n)
) * 100

scorecard = scorecard.merge(fm[['amfi_code', 'scheme_name', 'fund_house']], on='amfi_code')
scorecard = scorecard.sort_values('Score', ascending=False)
scorecard[['scheme_name', 'Score', 'CAGR_3yr', 'Sharpe', 'Alpha', 'Max_DD', 'expense_ratio_pct']].head(10)

In [ ]:
# Save scorecard
out_cols = ['amfi_code', 'scheme_name', 'fund_house', 'Score', 'CAGR_3yr', 'Sharpe', 'Alpha', 'Max_DD', 'expense_ratio_pct']
scorecard[out_cols].to_csv(REPO / 'fund_scorecard.csv', index=False)
print('Saved fund_scorecard.csv')

## 8. Benchmark Comparison Chart

In [ ]:
top5_codes = scorecard.head(5)['amfi_code'].tolist()

cutoff_3yr = common_dates[-756] if len(common_dates) >= 756 else common_dates[0]
nifty50 = bench[bench['index_name'] == 'NIFTY50'][['date', 'close_value']].copy().sort_values('date').set_index('date')
nifty100_level = bench[bench['index_name'] == 'NIFTY100'][['date', 'close_value']].copy().sort_values('date').set_index('date')

common_3yr = nav_pivot.loc[cutoff_3yr:].index.intersection(nifty50.index).intersection(nifty100_level.index)

fig, ax = plt.subplots(figsize=(14, 8))

for code in top5_codes:
    fund_cum = (1 + daily_returns.loc[common_3yr, code]).cumprod()
    name = fm[fm['amfi_code'] == code]['scheme_name'].values[0]
    ax.plot(fund_cum.index, fund_cum.values, label=name, linewidth=2)

n50_cum = (1 + nifty50.loc[common_3yr, 'close_value'].pct_change().dropna()).cumprod()
n100_cum = (1 + nifty100_level.loc[common_3yr, 'close_value'].pct_change().dropna()).cumprod()

ax.plot(n50_cum.index, n50_cum.values, label='NIFTY 50', linewidth=2, linestyle='--', color='gray')
ax.plot(n100_cum.index, n100_cum.values, label='NIFTY 100', linewidth=2, linestyle=':', color='black')

ax.set_title('Top 5 Funds vs Benchmarks (3-Year Cumulative Returns)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (1 = 0%)')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(REPO / 'benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tracking Error
fund_ret_3yr = daily_returns.loc[common_3yr, top5_codes]
n50_ret_3yr = nifty50.loc[common_3yr, 'close_value'].pct_change().dropna()
n100_ret_3yr = nifty100_level.loc[common_3yr, 'close_value'].pct_change().dropna()

print('Tracking Error (annualized, 3-year):')
for code in top5_codes:
    name = fm[fm['amfi_code'] == code]['scheme_name'].values[0]
    te_n50 = (fund_ret_3yr[code] - n50_ret_3yr).std() * np.sqrt(252)
    te_n100 = (fund_ret_3yr[code] - n100_ret_3yr).std() * np.sqrt(252)
    print(f'{name[:55]:55s} | TE vs N50: {te_n50:.4f} | TE vs N100: {te_n100:.4f}')

## Summary

| Metric | Key Finding |
|---|---|
| **Daily Returns** | Mean ~0.06%, std ~1.03%, min -5.8%, max +6.5% |
| **CAGR 3yr** | Ranges from -11.2% (Axis Small Cap) to +36.2% (Axis Midcap) |
| **Sharpe Ratio** | Best: Mirae Asset Large Cap (1.45), Kotak Flexicap (1.31) |
| **Sortino Ratio** | Best: Mirae Asset Large Cap (2.39), Kotak Flexicap (2.36) |
| **Alpha** | Many funds show positive alpha vs NIFTY 100 |
| **Max Drawdown** | Worst: SBI Small Cap Direct (-52.6%), Axis Small Cap (-51.7%) |
| **Top Scorecard** | ICICI Pru Midcap (82.0), HDFC Mid-Cap Opp (79.5), Axis Midcap (78.3) |

### Deliverables
- `fund_scorecard.csv` — ranked scorecard for all 40 funds
- `alpha_beta.csv` — alpha, beta, R² for all funds
- `benchmark_comparison.png` — top 5 funds vs NIFTY 50 & NIFTY 100

In [ ]:
# Save alpha_beta.csv
alpha_beta_df[['amfi_code', 'scheme_name', 'fund_house', 'Alpha', 'Beta', 'R_squared', 'p_value']].to_csv(REPO / 'alpha_beta.csv', index=False)
print('Saved alpha_beta.csv')
print('All deliverables generated.')